# 03 - Train Models

This notebook trains the three required explainable ML models for CleanSight AI:

1. Decision Tree Classifier for cleanliness prediction.
2. Isolation Forest for anomaly detection.
3. Linear Regression for next dust forecasting.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd()
if cwd.name == "notebooks":
    BASE_DIR = cwd.parent
elif (cwd / "model building").exists():
    BASE_DIR = cwd / "model building"
else:
    BASE_DIR = cwd

sys.path.append(str(BASE_DIR / "src"))
DATA_PATH = BASE_DIR / "data" / "processed" / "cleansight_processed_dataset.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Processed dataset:", DATA_PATH)
print("Model directory:", MODEL_DIR)

Processed dataset: /Users/vinushan/Documents/VAUED-Project/model building/data/processed/cleansight_processed_dataset.csv
Model directory: /Users/vinushan/Documents/VAUED-Project/model building/models


In [2]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print("Rows:", len(df))
df.head()

Rows: 1171


,session_id,house_id,room_id,session_type,device_id,start_time,end_time,status,reading_interval_seconds,total_readings,...,air_quality_rolling_mean_3,temperature_rolling_mean_3,humidity_rolling_mean_3,dust_lag_1,air_quality_lag_1,temperature_lag_1,humidity_lag_1,next_dust,cleanliness_label,label_source
0,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,328.040000,28.590000,66.280000,218.74,328.04,28.59,66.28,191.38,dirty,weak_sensor_rule
1,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,312.330000,28.605000,67.755000,218.74,328.04,28.59,66.28,161.80,dirty,weak_sensor_rule
2,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,293.356667,28.543333,68.240000,191.38,296.62,28.62,69.23,136.90,dirty,weak_sensor_rule
3,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,254.706667,28.500000,69.756667,161.80,255.41,28.42,69.21,131.52,needs_attention,weak_sensor_rule
4,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,225.493333,28.496667,69.043333,136.90,212.09,28.46,70.83,103.80,needs_attention,weak_sensor_rule


## Model 1 - Decision Tree Classifier

Purpose: predict `cleanliness_label` as `clean`, `needs_attention`, or `dirty`.

Why this model: it is explainable, works well with small tabular sensor datasets, and provides feature importance for viva/demo.

## Model 2 - Isolation Forest

Purpose: detect abnormal sensor patterns such as sudden dust spikes or unusual air quality values.

Why this model: it does not require manual anomaly labels, which is useful for IoT sensor projects with limited labeled data.

## Model 3 - Linear Regression

Purpose: predict `next_dust` for short-term trend forecasting.

Why this model: it is simple, explainable, and enough for showing temporal trend analysis with limited sensor data.

In [3]:
from train import (
    train_decision_tree,
    train_isolation_forest,
    train_linear_regression,
    save_models,
)

decision_tree_model, X_cls_test, y_cls_test = train_decision_tree(df, random_state=42)
isolation_forest_model, anomaly_training_df = train_isolation_forest(df, random_state=42)
linear_regression_model, X_reg_test, y_reg_test = train_linear_regression(df, random_state=42)

feature_map = save_models(
    decision_tree_model,
    isolation_forest_model,
    linear_regression_model,
    model_dir=str(MODEL_DIR),
)

print("Saved models and feature lists.")
feature_map

Saved models and feature lists.


{'decision_tree_cleanliness': ['dust',
  'air_quality',
  'temperature',
  'humidity',
  'hour_of_day',
  'dust_rolling_mean_3',
  'air_quality_rolling_mean_3'],
 'isolation_forest_anomaly': ['dust',
  'air_quality',
  'temperature',
  'humidity'],
 'linear_regression_dust_forecast': ['dust_lag_1',
  'air_quality_lag_1',
  'temperature_lag_1',
  'humidity_lag_1',
  'hour_of_day']}

## Saved model files

The trained models are saved as `.pkl` files so the evaluation notebook, inference demo, and future backend/dashboard integration can load them without retraining.

In [4]:
for path in sorted(MODEL_DIR.glob("*")):
    print(path.name)

cleansight_tinyml_metadata.json
cleansight_tinyml_models.h
decision_tree_cleanliness.pkl
isolation_forest_anomaly.pkl
linear_regression_dust_forecast.pkl
model_features.json
